# Streaming Content Moderation with Qwen3Guard and OpenVINO

Export Qwen3Guard-Stream-0.6B to OpenVINO, moderate user prompts and streaming responses, guard Qwen3 generation, and explore the results in Gradio.

Model cards: [`Qwen/Qwen3Guard-Stream-0.6B`](https://huggingface.co/Qwen/Qwen3Guard-Stream-0.6B).

This tutorial demonstrates how to:

- convert the model to OpenVINO IR with Optimum Intel;
- moderate a user prompt and inspect risk levels and categories;
- preserve KV-cache state while moderating a fixed response token by token;
- guard live Qwen3-0.6B generation and launch an interactive Gradio demo.

⚠️ **EXPERIMENTAL NOTEBOOK**

This notebook demonstrates a model that has not been fully validated with OpenVINO and is using a custom branch of optimum-intel. It may be fully supported and validated in the future.

The guard and the additional [Qwen3-0.6B generator](https://huggingface.co/Qwen/Qwen3-0.6B) are Apache-2.0 models. The guard has four classification heads, not a language-model head: response risk (3), response category (8), query risk (3), and query category (9). Risk levels are Safe, Unsafe, and Controversial; Jailbreak is an input-only category. See the [Qwen3Guard project](https://github.com/QwenLM/Qwen3Guard) for the taxonomy. Predictions can be wrong and do not replace application-specific safety evaluation.



### Installation Instructions

This is a self-contained example that relies solely on its own code.

We recommend  running the notebook in a virtual environment. You only need a Jupyter server to start.
For details, please refer to [Installation Guide](https://github.com/openvinotoolkit/openvino_notebooks/blob/latest/README.md#-installation-guide).

<img referrerpolicy="no-referrer-when-downgrade" src="https://static.scarf.sh/a.png?x-pxid=5b5a4db0-7875-4bfb-bdbd-01698b5b1a77&file=notebooks/qwen3guard-stream/qwen3guard-stream.ipynb" />

#### Table of contents:

- [Prerequisites](#Prerequisites)
    - [Installation](#Installation)
- [Select model and export options](#Select-model-and-export-options)
- [Select inference device](#Select-inference-device)
- [Convert the model to OpenVINO IR](#Convert-the-model-to-OpenVINO-IR)
- [Inspect the guard and moderate a prompt](#Inspect-the-guard-and-moderate-a-prompt)
- [Moderate a fixed response token by token](#Moderate-a-fixed-response-token-by-token)
- [Guard live Qwen3 generation](#Guard-live-Qwen3-generation)
- [Interactive demo](#Interactive-demo)
- [Clean up](#Clean-up)



## Prerequisites
[back to top ⬆️](#Table-of-contents:)

### Installation
[back to top ⬆️](#Table-of-contents:)

Install the required dependencies:

- This day-0 example pins [Optimum Intel PR 2039](https://github.com/huggingface/optimum-intel/pull/2039) at commit `3c540359661261a6edbc2a9bb9fc3df05230901f` for multi-head token classification, stateful export, and `OVGuard`. Replace the pin only after validating a supporting release.
- Transformers 4.57.6 is required by the checkpoint's custom code; Transformers 5 is not supported by this example.
- OpenVINO 2026.4.1 is the initial runtime baseline. This notebook uses Optimum Intel, because the earlier [GenAI moderation proposal](https://github.com/openvinotoolkit/openvino.genai/pull/4480) was closed without merging.

Restart the notebook kernel after installing or changing these packages.

In [ ]:
import urllib.request
from pathlib import Path

# Helpers come from `utils` of the cloned repository, otherwise from GitHub; local copies are overwritten
helper_files = ["notebook_utils.py", "cmd_helper.py", "pip_helper.py"]
repo_utils = Path("../../utils")
in_repo = (repo_utils / "notebook_utils.py").is_file()
if in_repo:
    helper_contents = {helper_file: (repo_utils / helper_file).read_bytes() for helper_file in helper_files}
else:
    try:
        helper_contents = {}
        for helper_file in helper_files:
            url = f"https://raw.githubusercontent.com/openvinotoolkit/openvino_notebooks/latest/utils/{helper_file}"
            with urllib.request.urlopen(url, timeout=10) as response:
                helper_contents[helper_file] = response.read()
    except OSError as error:
        if any(not Path(helper_file).is_file() for helper_file in helper_files):
            raise
        print(f"Could not update helper files, using the existing copies: {error}")
        helper_contents = {}

for helper_file, content in helper_contents.items():
    Path(helper_file).write_bytes(content)

from notebook_utils import setup_notebook_venv

# Requirements installed below go to a notebook-specific virtual environment in the `.venv` folder
setup_notebook_venv()

In [ ]:
from pip_helper import pip_install

pip_install(
    "-q",
    "git+https://github.com/huggingface/optimum-intel.git@3c540359661261a6edbc2a9bb9fc3df05230901f",
    "transformers==4.57.6",
    "torch==2.8.0",
    "nncf==3.4.0",
    "openvino==2026.4.1",
    "openvino-tokenizers==2026.4.1.0",
    "--extra-index-url",
    "https://download.pytorch.org/whl/cpu",
)
pip_install("-q", "gradio>=5,<6", "ipywidgets", "accelerate", "einops")

import ipywidgets as widgets
from IPython.display import display

# Read more about telemetry collection at https://github.com/openvinotoolkit/openvino_notebooks?tab=readme-ov-file#-telemetry
from notebook_utils import collect_telemetry

collect_telemetry("qwen3guard-stream.ipynb")

## Select model and export options
[back to top ⬆️](#Table-of-contents:)

This notebook uses FP16 weights. INT8 weight compression is not offered because it changed a moderation decision in a small comparison against the original FP32 model. Ordinary Qwen3 LLM compression settings are not a guard-quality guarantee. Source snapshots and exported models are stored below `models`.

In [ ]:
model_ids = ["Qwen/Qwen3Guard-Stream-0.6B"]

model_id_widget = widgets.Dropdown(options=model_ids, value=model_ids[0], description="Model:")
display(model_id_widget)

## Select inference device
[back to top ⬆️](#Table-of-contents:)

Select CPU or an available Intel GPU for both models. Native NPU moderation needs separate [NPUW work](https://github.com/openvinotoolkit/openvino/pull/39353) that is not yet merged, so NPU and AUTO are not offered. Classification requests FP32 inference precision to make the numerical baseline explicit; this is separate from the weight storage format.

In [ ]:
from notebook_utils import device_widget

device = device_widget("CPU", exclude=["AUTO", "NPU"])
device

In [ ]:
GUARD_REVISION = "74e1479150e9029d6778993f00491108323bb6f8"
GENERATOR_REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
model_id = model_id_widget.value
generator_id = "Qwen/Qwen3-0.6B"
precision = "fp16"
model_dir = Path("models") / f"{Path(model_id).name}-{GUARD_REVISION[:8]}-{precision}-ov"
generator_dir = Path("models") / f"{Path(generator_id).name}-{GENERATOR_REVISION[:8]}-{precision}-ov"
print(f"Model: {model_id}, weights: {precision}, OpenVINO IR directory: {model_dir}")

## Convert the model to OpenVINO IR
[back to top ⬆️](#Table-of-contents:)

Download the pinned source snapshot, including the reviewed custom Python code, then use `optimum-cli export openvino`. The `token-classification-with-past` task preserves all four heads and makes the KV cache stateful. This is classification, not `text-generation`. The source revision includes the upstream streaming cache-reuse fix. CI substitutes local random models with the same guard heads and Qwen3 generator architecture, without downloading pretrained weights.

In [ ]:
from cmd_helper import optimum_cli
from huggingface_hub import snapshot_download

additional_args = {"weight-format": precision, "task": "token-classification-with-past", "trust-remote-code": ""}
if precision == "int8":
    additional_args["sym"] = ""

if all((model_dir / filename).is_file() for filename in ("openvino_model.xml", "openvino_model.bin", "config.json")):
    print(f"Using existing OpenVINO model from {model_dir.resolve()}")
else:
    source_dir = (
        model_id
        if Path(model_id).is_dir()
        else snapshot_download(repo_id=model_id, revision=GUARD_REVISION, local_dir=Path("models") / f"{Path(model_id).name}-{GUARD_REVISION[:8]}-source")
    )
    optimum_cli(source_dir, model_dir, additional_args=additional_args)

## Inspect the guard and moderate a prompt
[back to top ⬆️](#Table-of-contents:)

`OVGuard` selects the query heads for user input and the response heads for assistant output. A category is predicted even for Safe content; display it as a violation category only for non-Safe predictions. This example blocks Unsafe only; Controversial remains visible for the application to decide.

The guard uses FP32 inference. On CPU, FP32 KV-cache storage and disabled dynamic activation quantization keep incremental predictions consistent with full-prefix inference.

In [ ]:
from collections.abc import Iterator
from dataclasses import asdict
from functools import partial
import torch
from transformers import AutoTokenizer, PreTrainedTokenizerBase
from optimum.intel import OVGuard, OVGuardConfig, OVModelForCausalLM

MAX_INPUT_TOKENS = 512
guard_tokenizer = AutoTokenizer.from_pretrained(model_dir, trust_remote_code=True)
guard_ov_config = {"INFERENCE_PRECISION_HINT": "f32"}
if device.value == "CPU":
    guard_ov_config.update({"KV_CACHE_PRECISION": "f32", "DYNAMIC_QUANTIZATION_GROUP_SIZE": "0"})
guard = OVGuard.from_pretrained(
    model_dir,
    trust_remote_code=True,
    tokenizer=guard_tokenizer,
    device=device.value,
    ov_config=guard_ov_config,
)
probe = guard_tokenizer("Hello!", return_tensors="pt")
head_outputs = guard.model(**probe).logits
head_sizes = {"risk_level_logits": 3, "category_logits": 8, "query_risk_level_logits": 3, "query_category_logits": 9}
assert set(head_outputs) == set(head_sizes)
for name, classes in head_sizes.items():
    assert tuple(head_outputs[name].shape) == (*probe.input_ids.shape, classes)
guard.reset()
print("GUARD_HEADS:", {name: tuple(output.shape) for name, output in head_outputs.items()})


def moderate_prompt(text: str, guard: OVGuard, tokenizer: PreTrainedTokenizerBase, max_tokens: int) -> dict:
    if not text.strip():
        raise ValueError("Enter a non-empty user prompt.")
    ids = tokenizer.apply_chat_template([{"role": "user", "content": text}], add_generation_prompt=True)
    if len(ids) > max_tokens:
        raise ValueError(f"The formatted prompt exceeds {max_tokens} tokens.")
    try:
        return asdict(guard.moderate_prompt(text).verdicts[0])
    finally:
        guard.reset()


prompt = "How can I grow tomatoes on a balcony?"
prompt_verdict = moderate_prompt(prompt, guard=guard, tokenizer=guard_tokenizer, max_tokens=MAX_INPUT_TOKENS)
print("PROMPT_MODERATION:", prompt_verdict)

## Moderate a fixed response token by token
[back to top ⬆️](#Table-of-contents:)

Prefill the user turn once, then submit only new tokens while retaining the KV cache. The table includes assistant-header and end-of-turn tokens. Every verdict describes the accumulated conversation, not an isolated token. Reset the state between independent conversations. This is a diagnostic visualization of a simulated stream: it reports flags but deliberately does not stop or generate text.

In [ ]:
MAX_STREAM_TOKENS = 96


def moderate_stream(
    prompt: str, response: str, guard: OVGuard, tokenizer: PreTrainedTokenizerBase, max_input_tokens: int, max_stream_tokens: int
) -> Iterator[list]:
    if not prompt.strip() or not response.strip():
        raise ValueError("Enter both a user prompt and an assistant response.")
    messages = [{"role": "user", "content": prompt}]
    prompt_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=False, return_tensors="pt")
    full_ids = tokenizer.apply_chat_template(messages + [{"role": "assistant", "content": response}], add_generation_prompt=False, return_tensors="pt")
    prefix_length = prompt_ids.shape[1]
    if prefix_length > max_input_tokens:
        raise ValueError(f"The formatted prompt exceeds {max_input_tokens} tokens.")
    if not torch.equal(prompt_ids, full_ids[:, :prefix_length]):
        raise ValueError("The conversation template changed the user prefix; token IDs cannot be streamed safely.")
    if full_ids.shape[1] - prefix_length > max_stream_tokens:
        raise ValueError(f"The formatted response exceeds {max_stream_tokens} tokens; shorten the response.")
    rows = []
    guard.reset()
    try:
        guard.moderate_token_ids(prompt_ids, scope="prompt")
        for offset in range(full_ids.shape[1] - prefix_length):
            token_ids = full_ids[:, prefix_length + offset : prefix_length + offset + 1]
            verdict = guard.moderate_token_ids(token_ids, scope="response", token_offset=offset)[0][0]
            category = ", ".join(verdict.categories) if verdict.label != "Safe" else ""
            rows.append([repr(tokenizer.decode(token_ids[0])), verdict.label, verdict.score, category, verdict.flagged])
            yield list(rows)
    finally:
        guard.reset()


response = "Choose a sunny spot and water the plants regularly."
stream_rows = []
for stream_rows in moderate_stream(
    prompt, response, guard=guard, tokenizer=guard_tokenizer, max_input_tokens=MAX_INPUT_TOKENS, max_stream_tokens=MAX_STREAM_TOKENS
):
    print("STREAM_MODERATION:", stream_rows[-1])
assert stream_rows

## Guard live Qwen3 generation
[back to top ⬆️](#Table-of-contents:)

The next cell downloads and exports a second model, Qwen3-0.6B. It generates text; Qwen3Guard-Stream only classifies it. Direct token-ID moderation requires identical token vocabularies, not merely equal vocabulary sizes. We compare the two tokenizer mappings before generation. `prompt_mode="sync"` checks the input first; `emit_before_check=False` prevents unchecked chunks reaching a streamer. `on_violation="stop"` removes the flagged generated suffix. These controls do not guarantee that the classifier detects every unsafe response.

In [ ]:
generator_args = {"weight-format": precision, "task": "text-generation-with-past"}
if precision == "int8":
    generator_args["sym"] = ""
if not all((generator_dir / filename).is_file() for filename in ("openvino_model.xml", "openvino_model.bin", "config.json")):
    generator_source = (
        generator_id
        if Path(generator_id).is_dir()
        else snapshot_download(
            repo_id=generator_id,
            revision=GENERATOR_REVISION,
            local_dir=Path("models") / f"{Path(generator_id).name}-{GENERATOR_REVISION[:8]}-source",
        )
    )
    optimum_cli(generator_source, generator_dir, additional_args=generator_args)
generator_tokenizer = AutoTokenizer.from_pretrained(generator_dir)
if generator_tokenizer.get_vocab() != guard_tokenizer.get_vocab():
    raise ValueError("The generator and guard must use identical token vocabularies for direct token-ID moderation.")
generator = OVModelForCausalLM.from_pretrained(generator_dir, device=device.value)
print("Generator:", generator_id)

In [ ]:
MAX_NEW_TOKENS = 64


def generate_guarded(
    text: str, generator: OVModelForCausalLM, tokenizer: PreTrainedTokenizerBase, guard: OVGuard, max_input_tokens: int, max_new_tokens: int
) -> tuple[str, list]:
    if not text.strip():
        raise ValueError("Enter a non-empty user prompt.")
    inputs = tokenizer.apply_chat_template([{"role": "user", "content": text}], add_generation_prompt=True, enable_thinking=False, return_tensors="pt")
    if inputs.shape[1] > max_input_tokens:
        raise ValueError(f"The formatted prompt exceeds {max_input_tokens} tokens.")
    guard.reset()
    try:
        output = generator.generate(
            inputs,
            attention_mask=torch.ones_like(inputs),
            pad_token_id=tokenizer.eos_token_id,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            tokenizer=tokenizer,
            guard=guard,
            guard_config=OVGuardConfig(chunk_size=4, prompt_mode="sync", on_violation="stop", emit_before_check=False),
            return_dict_in_generate=True,
        )
        answer = tokenizer.decode(output.sequences[0, inputs.shape[1] :], skip_special_tokens=True)
        return answer, [asdict(verdict) for verdict in output.guard.verdicts]
    finally:
        guard.reset()


answer, generation_verdicts = generate_guarded(
    "Give two short tips for growing tomatoes.",
    generator=generator,
    tokenizer=generator_tokenizer,
    guard=guard,
    max_input_tokens=MAX_INPUT_TOKENS,
    max_new_tokens=MAX_NEW_TOKENS,
)
assert generation_verdicts
print("GUARDED_ANSWER:", answer)
print("GUARDED_VERDICTS:", generation_verdicts)
print("QWEN3GUARD_RESULT: PASS")

## Interactive demo
[back to top ⬆️](#Table-of-contents:)

Explore prompt checks, fixed-response streaming, and guarded generation. Requests share a serialized queue because the guard's KV cache belongs to one conversation at a time. Input lengths are bounded as in the examples above.

In [ ]:
from gradio_helper import make_demo

demo = make_demo(
    moderate_prompt=partial(moderate_prompt, guard=guard, tokenizer=guard_tokenizer, max_tokens=MAX_INPUT_TOKENS),
    moderate_stream=partial(moderate_stream, guard=guard, tokenizer=guard_tokenizer, max_input_tokens=MAX_INPUT_TOKENS, max_stream_tokens=MAX_STREAM_TOKENS),
    generate=partial(
        generate_guarded,
        generator=generator,
        tokenizer=generator_tokenizer,
        guard=guard,
        max_input_tokens=MAX_INPUT_TOKENS,
        max_new_tokens=MAX_NEW_TOKENS,
    ),
)
demo.launch()

## Clean up
[back to top ⬆️](#Table-of-contents:)

Uncomment and run the cell below to remove the exported models and downloaded files.

In [ ]:
# import shutil

# demo.close()
# shutil.rmtree(model_dir)
# shutil.rmtree(generator_dir)
# Source snapshots remain in the named models/*-source directories; remove them separately if no longer needed.